# Smart Parking Occupancy Detection

A computer-vision project that uses YOLOv8 to detect vehicles, maps detections to manually defined parking bays, and exports an annotated video plus frame-by-frame occupancy CSV.

**Environment:** Google Colab is recommended. The parking-bay coordinates are specific to the supplied camera view/video.

## 1. Install dependencies

In [ ]:
%pip install -q ultralytics

In [ ]:
from ultralytics import YOLO
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

print("Imports complete!")

## 2. Upload and inspect the parking video

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
video_path = "parking1.mp4"

cap = cv2.VideoCapture(video_path)
print("Video opened successfully:", cap.isOpened())
cap.release()

In [ ]:
cap = cv2.VideoCapture(video_path)

fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

cap.release()

duration = total_frames / fps if fps > 0 else 0

print("FPS:", fps)
print("Total frames:", total_frames)
print("Resolution:", width, "x", height)
print("Duration:", round(duration, 2), "seconds")

In [ ]:
cap = cv2.VideoCapture(video_path)
success, first_frame = cap.read()
cap.release()

if success:
    first_frame_rgb = cv2.cvtColor(first_frame, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(12, 7))
    plt.imshow(first_frame_rgb)
    plt.title("First video frame")
    plt.axis("off")
    plt.show()

In [ ]:
cap = cv2.VideoCapture(video_path)
frame_numbers = (79, 90)

fig, axes = plt.subplots(1, len(frame_numbers), figsize=(18, 5))
if len(frame_numbers) == 1:
    axes = [axes]

for ax, frame_number in zip(axes, frame_numbers):
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_number))
    success, frame = cap.read()

    if success:
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        ax.imshow(frame_rgb)
        ax.set_title(f"Frame {frame_number}")
    else:
        ax.set_title(f"Could not read {frame_number}")

    ax.axis("off")

cap.release()
plt.tight_layout()
plt.show()

## 3. Load the YOLO model and detect vehicles

In [ ]:
model = YOLO("yolov8n.pt")
print(model.names)

In [ ]:
cap = cv2.VideoCapture(video_path)
cap.set(cv2.CAP_PROP_POS_FRAMES, 80)
success, sample_frame = cap.read()
cap.release()

results = model.predict(
    source=sample_frame,
    classes=[2, 3, 5, 7],
    conf=0.20,
    imgsz = 960,
    verbose=False
)

result = results[0]
print("Vehicle detections:", len(result.boxes))

In [ ]:
annotated = result.plot()

plt.figure(figsize=(14, 8))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.title("YOLO detections on one frame")
plt.axis("off")
plt.show()

## 4. Define parking-bay regions

In [ ]:
bays = {
    "bay1":  [[85, 790],   [180, 785],  [148, 905],  [38, 905]],
    "bay2":  [[180, 785],  [283, 780],  [265, 905],  [148, 905]],
    "bay3":  [[283, 780],  [395, 775],  [395, 895],  [265, 905]],
    "bay4":  [[395, 775],  [506, 765],  [521, 885],  [395, 895]],
    "bay5":  [[506, 765],  [620, 755],  [652, 872],  [521, 885]],
    "bay6":  [[620, 755],  [734, 745],  [783, 857],  [652, 872]],
    "bay7":  [[734, 745],  [843, 733],  [907, 842],  [783, 857]],
    "bay8":  [[843, 733],  [948, 720],  [1023, 826], [907, 842]],
    "bay9":  [[948, 720],  [1045, 710], [1137, 810], [1023, 826]],
    "bay10": [[1045, 710], [1140, 700], [1237, 790], [1137, 810]],
    "bay11": [[1140, 700], [1226, 686], [1323, 769], [1237, 790]],
    "bay12": [[1226, 686], [1312, 672], [1409, 747], [1323, 769]],
}

print("Number of defined bays:", len(bays))

In [ ]:
vis = first_frame.copy()

for bay_name, points in bays.items():
    polygon = np.array(points, dtype=np.int32)
    cv2.polylines(vis, [polygon], isClosed=True, color=(0, 255, 0), thickness=2)

    center_x, center_y = polygon.mean(axis=0).astype(int)
    label = bay_name.replace("bay", "")
    cv2.putText(
        vis, label, (center_x, center_y),
        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2
    )

plt.figure(figsize=(16, 7))
plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
plt.title("Parking bay polygons")
plt.axis("on")
plt.show()

## 5. Map detected vehicles to parking bays

In [ ]:
def find_bay(px, py):
  for bay_name, points in bays.items():
    polygon = np.array(points, dtype=np.int32)

    inside = cv2.pointPolygonTest(
        polygon,
        (float(px), float(py)),
        False
    )

    if inside >=0:
      return bay_name

  return None

In [ ]:
bottom_point_raise = 0.25

for box in result.boxes:
    x1, y1, x2, y2 = box.xyxy[0].tolist()

    px = (x1 + x2) / 2
    py = y2 - bottom_point_raise * (y2 - y1)

    bay_name = find_bay(px, py)
    print(f"Point ({int(px)}, {int(py)}) -> {bay_name}")

In [ ]:
vis = sample_frame.copy()

for bay_name, points in bays.items():
    cv2.polylines(vis, [np.array(points, dtype=np.int32)], True, (0, 255, 0), 2)

for box in result.boxes:
    x1, y1, x2, y2 = box.xyxy[0].tolist()
    px = int((x1 + x2) / 2)
    py = int(y2 - bottom_point_raise * (y2 - y1))
    bay_name = find_bay(px, py)

    color = (0, 255, 0) if bay_name is not None else (0, 165, 255)
    cv2.circle(vis, (px, py), 7, color, -1)

plt.figure(figsize=(15, 7))
plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
plt.title("Vehicle reference points and parking bays")
plt.axis("on")
plt.show()

## 6. Calculate occupancy for a sample frame

In [ ]:
occupied_bays = set()

for box in result.boxes:
    x1, y1, x2, y2 = box.xyxy[0].tolist()
    px = (x1 + x2) / 2
    py = y2 - bottom_point_raise * (y2 - y1)

    bay_name = find_bay(px, py)
    if bay_name is not None:
        occupied_bays.add(bay_name)

total_bays = len(bays)
occupied_count = len(occupied_bays)
vacant_count = total_bays - occupied_count

print("Occupied bays:", sorted(occupied_bays))
print("Occupied:", occupied_count)
print("Vacant:", vacant_count)
print("Total:", total_bays)

In [ ]:
vis = sample_frame.copy()

for bay_name, points in bays.items():
    polygon = np.array(points, dtype=np.int32)
    occupied = bay_name in occupied_bays
    color = (0, 0, 255) if occupied else (0, 255, 0)

    cv2.polylines(vis, [polygon], True, color, 3)
    center_x, center_y = polygon.mean(axis=0).astype(int)
    cv2.putText(
        vis, bay_name.replace("bay", ""), (center_x, center_y),
        cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2
    )

cv2.putText(
    vis,
    f"Occupied: {occupied_count}  Vacant: {vacant_count}",
    (30, 50),
    cv2.FONT_HERSHEY_SIMPLEX,
    1.0,
    (255, 255, 255),
    2
)

plt.figure(figsize=(16, 7))
plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
plt.title("Single-frame parking occupancy")
plt.axis("off")
plt.show()

## 7. Process the full video and export results

In [ ]:
output_video = "smart_parking_output.mp4"
output_csv = "parking_occupancy.csv"

confidence_threshold = 0.25
vehicle_classes = [2, 3, 5, 7]  # car, motorcycle, bus, truck in COCO

In [ ]:
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    raise RuntimeError(f"Could not open video: {video_path}")

fps = cap.get(cv2.CAP_PROP_FPS)
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

if fps <= 0 or frame_width <= 0 or frame_height <= 0:
    cap.release()
    raise RuntimeError("Video metadata is invalid.")

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(
    output_video, fourcc, fps, (frame_width, frame_height)
)

if not writer.isOpened():
    cap.release()
    raise RuntimeError("Could not create the output video.")

print("Frames to process:", frame_count)
print("Output video:", output_video)

In [ ]:

import csv
import cv2
import numpy as np

# Start fresh and make sure the video starts at frame zero.
cap.release()
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    raise RuntimeError(f"Could not open video: {video_path}")

# Reset the output writer if it was previously used.
writer.release()

fps = cap.get(cv2.CAP_PROP_FPS)
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(
    output_video,
    fourcc,
    fps,
    (frame_width, frame_height)
)

if not writer.isOpened():
    cap.release()
    raise RuntimeError("Could not create the output video.")

frame_index = 0
csv_rows = []

try:
    while True:
        success, frame = cap.read()

        if not success:
            break

        # Run YOLO on the original, full-size frame.
        detections = model.predict(
            source=frame,
            classes=vehicle_classes,
            conf=confidence_threshold,
            verbose=False
        )[0]

        occupied_this_frame = set()
        relevant_boxes = []

        # Match detected vehicles to the defined parking bays.
        for box in detections.boxes:
            x1, y1, x2, y2 = box.xyxy[0].tolist()

            # Reference point near the bottom-centre of the vehicle.
            px = (x1 + x2) / 2
            py = y2 - bottom_point_raise * (y2 - y1)

            bay_name = find_bay(px, py)

            # Keep only detections matched to a parking bay.
            if bay_name is not None:
                occupied_this_frame.add(bay_name)
                relevant_boxes.append(box)

        # Calculate occupancy.
        occupied_count = len(occupied_this_frame)
        vacant_count = len(bays) - occupied_count

        # Draw the original parking-bay polygons.
        for bay_name, points in bays.items():
            polygon = np.array(points, dtype=np.int32)

            occupied = bay_name in occupied_this_frame
            color = (0, 0, 255) if occupied else (0, 255, 0)

            cv2.polylines(
                frame, [polygon], isClosed=True,
                color=color, thickness=2
            )

            center_x, center_y = polygon.mean(axis=0).astype(int)

            cv2.putText(
                frame,
                bay_name.replace("bay", ""),
                (center_x, center_y),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                color,
                2
            )

        # Draw ONLY the relevant vehicle bounding boxes.
        for box in relevant_boxes:
            x1, y1, x2, y2 = map(
                int, box.xyxy[0].tolist()
            )

            class_id = int(box.cls[0])
            confidence = float(box.conf[0])

            label = (
                f"{model.names[class_id]} "
                f"{confidence:.2f}"
            )

            # Orange bounding box and label.
            color = (0, 165, 255)

            cv2.rectangle(
                frame,
                (x1, y1),
                (x2, y2),
                color,
                2
            )

            cv2.putText(
                frame,
                label,
                (x1, max(y1 - 8, 20)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                color,
                2
            )

        # Display the occupancy summary.
        cv2.putText(
            frame,
            f"Occupied: {occupied_count} | Vacant: {vacant_count}",
            (25, 45),
            cv2.FONT_HERSHEY_SIMPLEX,
            1.0,
            (255, 255, 255),
            2
        )

        # Save the annotated frame to the output video.
        writer.write(frame)

        # Collect one CSV record per frame.
        csv_rows.append({
            "frame": frame_index,
            "occupied": occupied_count,
            "vacant": vacant_count,
            "total_bays": len(bays),
            "occupied_bay_names": ";".join(
                sorted(occupied_this_frame)
            )
        })

        frame_index += 1

        if frame_index % 50 == 0:
            print(f"Processed {frame_index} frames...")

finally:
    cap.release()
    writer.release()

print("Finished processing!")
print("Frames processed:", frame_index)
print("Output video:", output_video)

# Save the CSV results.
with open(
    output_csv, "w", newline="", encoding="utf-8"
) as csv_file:
    fieldnames = [
        "frame",
        "occupied",
        "vacant",
        "total_bays",
        "occupied_bay_names"
    ]

    csv_writer = csv.DictWriter(
        csv_file, fieldnames=fieldnames
    )
    csv_writer.writeheader()
    csv_writer.writerows(csv_rows)

print("Saved CSV:", output_csv)
print("CSV rows:", len(csv_rows))


In [ ]:
with open(output_csv, "w", newline="", encoding="utf-8") as csv_file:
    fieldnames = [
        "frame", "occupied", "vacant", "total_bays", "occupied_bay_names"
    ]
    csv_writer = csv.DictWriter(csv_file, fieldnames=fieldnames)
    csv_writer.writeheader()
    csv_writer.writerows(csv_rows)

print("Saved CSV:", output_csv)
print("Saved video:", output_video)
print("CSV rows:", len(csv_rows))

## 8. Inspect the occupancy data

In [ ]:
import pandas as pd

occupancy_df = pd.read_csv(output_csv)
display(occupancy_df)

## 9. Visualize occupancy over time

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(occupancy_df["frame"], occupancy_df["occupied"])
plt.xlabel("Frame number")
plt.ylabel("Occupied bays")
plt.title("Parking occupancy over time")
plt.grid(True, alpha=0.3)
plt.show()

## 10. Preview the annotated output video

In [ ]:
from IPython.display import Video, display

display(Video(output_video, embed=True))

## 11. Validate the exported video

In [ ]:
!ffprobe -v error -show_entries stream=codec_name,width,height,avg_frame_rate -show_entries format=duration,size -of default=noprint_wrappers=1 "{output_video}"

## 12. Check output files

In [ ]:
import os

print("File exists:", os.path.exists(output_video))

if os.path.exists(output_video):
    print("File size (MB):", round(
        os.path.getsize(output_video) / (1024 * 1024), 2
    ))

print("Frames processed:", frame_index)
print("CSV rows collected:", len(csv_rows))